In [1]:
import os

### Install Libraries

In [7]:
!pip install -q youtube-transcript-api langchain-community langchain-openai langchain \
               faiss-cpu tiktoken python-dotenv

In [13]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

### Indexing (Document Ingestion)

In [24]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi()
    transcript=transcript_list.fetch(video_id)
    # Flatten it to plain text
    ts=""
    for snippet in transcript:
     ts+=snippet.text
    # print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [20]:
transcript

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='the following is a conversation with', start=0.08, duration=3.44), FetchedTranscriptSnippet(text='demus hasabis', start=1.76, duration=4.96), FetchedTranscriptSnippet(text='ceo and co-founder of deepmind', start=3.52, duration=5.119), FetchedTranscriptSnippet(text='a company that has published and builds', start=6.72, duration=4.48), FetchedTranscriptSnippet(text='some of the most incredible artificial', start=8.639, duration=4.561), FetchedTranscriptSnippet(text='intelligence systems in the history of', start=11.2, duration=4.8), FetchedTranscriptSnippet(text='computing including alfred zero that', start=13.2, duration=3.68), FetchedTranscriptSnippet(text='learned', start=16.0, duration=2.96), FetchedTranscriptSnippet(text='all by itself to play the game of gold', start=16.88, duration=4.559), FetchedTranscriptSnippet(text='better than any human in the world and', start=18.96, duration=5.6), FetchedTranscriptSnippet(text='alph

In [25]:
ts

"the following is a conversation withdemus hasabisceo and co-founder of deepminda company that has published and buildssome of the most incredible artificialintelligence systems in the history ofcomputing including alfred zero thatlearnedall by itself to play the game of goldbetter than any human in the world andalpha fold two that solved proteinfoldingboth tasks considered nearly impossiblefor a very long timedemus is widely considered to be one ofthe most brilliant and impactful humansin the history of artificialintelligence and science and engineeringin generalthis was truly an honor and a pleasurefor me to finally sit down with him forthis conversation and i'm sure we willtalk many times again in the futurethis is the lex friedman podcast tosupport it please check out our sponsorsin the description and now dear friendshere's demishassabislet's start with a bit of a personalquestionam i an ai program you wrote tointerview people until i get good enoughto interview youwell i'll be im

### Text Splitting

In [26]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.create_documents([ts])

In [27]:
len(chunks)

163

In [28]:
chunks[100]

Document(metadata={}, page_content="functional than umrunning those simulations would bedo you think one day ai may allow us todo something like basically crack openphysics so do something like travelfaster than the speed of lightmy ultimate aim has always been with aiisum the reason i am personally working onai for my whole life it was to build atool to help us understand stand theuniverse so i wanted to and that meansphysics really and the nature of realitysoumuh i don't think we have systems thatare capable of doing that yet but whenwe get towards agi i think um that's oneof the first things i think we shouldapply agi toi would like to test the limits ofphysics and our knowledge of physicsthere's so many things we don't knowthere's one thing i find fascinatingabout science and you know as a hugeproponent of the scientific method asbeing one of the greatest ideashumanity's ever had and allowed us toprogress with our knowledgebut i think as a true scientist i thinkwhat you find is the

### Embedding Generation & Storing in Vector DB

In [29]:
!pip install langchain-huggingface sentence-transformers

In [30]:
# Local execution (Requires: pip install langchain-huggingface sentence-transformers)
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vector = embeddings.embed_query("Hello world")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [32]:
vector_store.index_to_docstore_id

{0: 'efff6a66-ef7e-4405-86fe-05879abcbb3c',
 1: 'd4556395-02ab-4721-a91e-9d4aff66d085',
 2: '12993527-9e15-40d6-85ba-6e68b80916e2',
 3: '54e66e2a-f813-4e12-b0c4-12e846cd3652',
 4: '5f6bea56-cc27-48ea-853a-d3799cc73175',
 5: 'fd1f81eb-a870-450c-8fe1-333caa54a704',
 6: '4ccda496-3a53-453c-b793-9620234e7534',
 7: '06d3654b-7cf6-4456-a4fd-2eaaf67c7c66',
 8: '3d4d74ad-a8dd-4c89-a372-594a971acb4e',
 9: 'd8355393-f061-426e-8fa2-0ddb2416b26e',
 10: 'be83311c-b24b-4225-9e99-c19c35476f9f',
 11: '047e93ed-cd32-4717-b9de-510c10b9c300',
 12: 'e44474d0-67ad-422b-abc6-79646d9fae43',
 13: '8579373e-b0f4-401c-839b-1e4bba96a6e8',
 14: '57101916-de0a-4f1d-9ca7-250cde42800e',
 15: '4194c66f-47c6-49a0-8816-dab6f6d511a7',
 16: '3a40e3c3-82e8-433e-82f8-31d3f7a5f6ef',
 17: '929a7fb1-6234-486b-8ae0-71854fe718e1',
 18: '0462e196-1d67-41db-a09a-41fda52a70bf',
 19: '9801c311-8655-46b7-872a-f0fa34b9bffd',
 20: 'dc373400-38bb-4a5d-b126-9b03a0a34cf7',
 21: 'd905c0c4-17f7-4164-bf23-b00043de6fe4',
 22: 'fcdc0f72-f47b-

In [34]:
vector_store.get_by_ids(['8e9b6c2d-a12d-4754-89e1-2efd3e087ee3'])

[Document(id='8e9b6c2d-a12d-4754-89e1-2efd3e087ee3', metadata={}, page_content="solve this small puzzle of aconversation with me today it's truly anhonor and a pleasure thank you thank youi really enjoyed it thanks lexthanks for listening to thisconversation with demas establish tosupport this podcast please check outour sponsors in the descriptionand now let me leave you with some wordsfrom edskar dykstracomputer science is no more aboutcomputersthan astronomy is about telescopesthank you for listening and hope to seeyou next time")]

### Retrieval

In [37]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [38]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fc051923020>, search_kwargs={'k': 4})

In [41]:
retriever.invoke('What is deepmind')

[Document(id='efff6a66-ef7e-4405-86fe-05879abcbb3c', metadata={}, page_content="the following is a conversation withdemus hasabisceo and co-founder of deepminda company that has published and buildssome of the most incredible artificialintelligence systems in the history ofcomputing including alfred zero thatlearnedall by itself to play the game of goldbetter than any human in the world andalpha fold two that solved proteinfoldingboth tasks considered nearly impossiblefor a very long timedemus is widely considered to be one ofthe most brilliant and impactful humansin the history of artificialintelligence and science and engineeringin generalthis was truly an honor and a pleasurefor me to finally sit down with him forthis conversation and i'm sure we willtalk many times again in the futurethis is the lex friedman podcast tosupport it please check out our sponsorsin the description and now dear friendshere's demishassabislet's start with a bit of a personalquestionam i an ai program you 

### Augmentation

In [43]:
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY here"

In [51]:
!pip install -q langchain-groq
from langchain_groq import ChatGroq

llm = ChatGroq(model_name="llama-3.3-70b-versatile",
temperature=0.2)

In [45]:
prompt=PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transcript context itself.
    If the content is insufficient, just say i dont know.

    {context}
    Question:{question}
    """,
    input_variables=['context','question']
)

In [46]:
question="is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs=retriever.invoke(question)
print(retrieved_docs)

[Document(id='959f02b2-7714-42db-89bd-3897ac7e47e4', metadata={}, page_content="deep rlso it's doing control of hightemperature plasmas can you explain thisworkand uh can ai eventually solve nuclearfusionit's been very fun last year or two andvery productive because we've beentaking off a lot of mydream projects if you like of thingsthat i've collected over the years ofareas of science that i would like to ithink could be very transformative if wehelped accelerate and uh reallyinteresting problems scientificchallenges in of themselvesthis is energy so energy yes exactly soenergy and climate so we talked aboutdisease and biology as being one of thebiggest places i think ai can help withi think energy and climate uh is anotherone so maybe they would be my top two umand fusion is one one area i think aican help with now fusion has manychallenges mostly physics materialscience and engineering challenges aswell to build these massive fusionreactors and contain the plasma and whatwe try to d

In [47]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"deep rlso it's doing control of hightemperature plasmas can you explain thisworkand uh can ai eventually solve nuclearfusionit's been very fun last year or two andvery productive because we've beentaking off a lot of mydream projects if you like of thingsthat i've collected over the years ofareas of science that i would like to ithink could be very transformative if wehelped accelerate and uh reallyinteresting problems scientificchallenges in of themselvesthis is energy so energy yes exactly soenergy and climate so we talked aboutdisease and biology as being one of thebiggest places i think ai can help withi think energy and climate uh is anotherone so maybe they would be my top two umand fusion is one one area i think aican help with now fusion has manychallenges mostly physics materialscience and engineering challenges aswell to build these massive fusionreactors and contain the plasma and whatwe try to do whenever we go into a newfieldto apply our systems is we look for umwe talk\n

In [48]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [49]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context itself.\n    If the content is insufficient, just say i dont know.\n\n    deep rlso it's doing control of hightemperature plasmas can you explain thisworkand uh can ai eventually solve nuclearfusionit's been very fun last year or two andvery productive because we've beentaking off a lot of mydream projects if you like of thingsthat i've collected over the years ofareas of science that i would like to ithink could be very transformative if wehelped accelerate and uh reallyinteresting problems scientificchallenges in of themselvesthis is energy so energy yes exactly soenergy and climate so we talked aboutdisease and biology as being one of thebiggest places i think ai can help withi think energy and climate uh is anotherone so maybe they would be my top two umand fusion is one one area i think aican help with now fusion has manychallenges mostly physics materialscience and eng

### Generation

In [52]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of nuclear fusion is discussed in this video. 

The discussion revolves around how AI can help solve nuclear fusion, which has many challenges, mostly physics, material science, and engineering challenges. The speaker mentions that they have made progress in this area by using AI to control high-temperature plasmas, which is a crucial aspect of fusion. They published a paper in Nature last year, where they were able to hold the plasma in specific shapes for a record amount of time. 

The speaker also talks about collaborating with fusion startups to identify the next problems to tackle in the fusion area and how they are looking to address bottleneck problems that are still stopping fusion from working today. Specifically, they mention plasma control as a problem that AI can help with, as plasma is unstable and needs to be predicted ahead of time to be contained. 

Additionally, the speaker touches on the topic of modeling and simulating the quantum mechanical behavior o

### Building a chain

1. RunnableParallel to run task in parallel
2. RunnableLambda to set it into context, pass the working function there as an arg
3. RunnablePassThrough since for question we get question in i/p and we have to pass that itself in the o/p

In [53]:
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [54]:
def format_docs(retrieved_docs):
  context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [55]:
parallel_chain=RunnableParallel(
    {
        'context':retriever | RunnableLambda(format_docs),
        'question':RunnablePassthrough()
    }
)

In [56]:
parallel_chain.invoke('Who is demis?')

{'context': "the following is a conversation withdemus hasabisceo and co-founder of deepminda company that has published and buildssome of the most incredible artificialintelligence systems in the history ofcomputing including alfred zero thatlearnedall by itself to play the game of goldbetter than any human in the world andalpha fold two that solved proteinfoldingboth tasks considered nearly impossiblefor a very long timedemus is widely considered to be one ofthe most brilliant and impactful humansin the history of artificialintelligence and science and engineeringin generalthis was truly an honor and a pleasurefor me to finally sit down with him forthis conversation and i'm sure we willtalk many times again in the futurethis is the lex friedman podcast tosupport it please check out our sponsorsin the description and now dear friendshere's demishassabislet's start with a bit of a personalquestionam i an ai program you wrote tointerview people until i get good enoughto interview youwel

In [57]:
parser=StrOutputParser()

In [58]:
main_chain=parallel_chain|prompt|llm|parser

In [61]:
main_chain.invoke('Can you summarise the video')

"The conversation is between Demis Hassabis and an interviewer, possibly an AI program. They discuss the Turing test, a benchmark for measuring intelligence, and whether the interviewer, if it's an AI, would be able to pass the test. Demis mentions that the Turing test was more of a thought experiment and not a rigorous formal test. They also talk about the limitations of language as a modality for expressing capabilities and the importance of other modalities like visual, robotics, and body language. The conversation touches on the idea of a meta Turing test, where the interviewer is an AI trying to interview Demis, and the Heisenberg uncertainty principle, where knowing the interviewer is an AI could change its behavior."

### Gradio UI

In [62]:
import gradio as gr

def answer_question(question):
    return main_chain.invoke(question)

iface = gr.Interface(
    fn=answer_question,
    inputs=gr.Textbox(lines=2, placeholder="Enter your question here..."),
    outputs="text",
    title="YouTube Transcript QA Chatbot",
    description="Ask questions about the YouTube video transcript."
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://20caae3a9794d6a625.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
